# Импорты и параметры

In [3]:
import os
import sys
import requests
import pandas as pd
from transformers import AutoTokenizer
from pathlib import Path
import asyncio
import random
import time

from openai import (
    AsyncOpenAI,
    APIConnectionError,
    APITimeoutError,
    BadRequestError,
    InternalServerError,
    RateLimitError
)

sys.path.append(str(Path.cwd().parent))

from long_input import prepare_product_input
from io_utils import load_jsonl
from generation import GenerationResult
from validation import (
    ValidationReason,
    ResponseValidationError,
    parse_and_validate
)
from prompts import (
    build_messages,
    build_retry_message
)

In [4]:
GENERATION_PARAMS = {
    "temperature": 0.5,
    "top_p": 0.9,
    "repetition_penalty": 1.1,
    "max_tokens": 250,
    "seed": 42
}

MAX_ATTEMPTS = 2
MAX_API_ATTEMPTS = 4

API_SAMPLE_SIZE = 20

# Выбор модели

In [5]:
MODELS_URL = "https://openrouter.ai/api/v1/models"

REQUIRED_PARAMETERS = {
    "temperature",
    "top_p",
    "repetition_penalty",
    "max_tokens",
    "seed",
    "response_format",
}

MIN_CONTEXT_LENGTH = 4096
N_CANDIDATES = 5

In [6]:
def model_selection(
        required_parameters: set[str],
        k: int = 5,
        min_context_length: int = 4096
) -> list[dict]:
    """Отбирает бесплатные модели OpenRouter, подходящие для сравнения."""

    response = requests.get(
        MODELS_URL,
        timeout=30,
    )
    response.raise_for_status()

    models = response.json()["data"]

    candidates = []

    for model in models:
        pricing = model.get("pricing", {})
        architecture = model.get("architecture", {})
        supported_parameters = set(
            model.get("supported_parameters", [])
        )
        reasoning = model.get("reasoning") or {}

        if (
            float(pricing.get("prompt", 1)) != 0
            or float(pricing.get("completion", 1)) != 0
        ):
            continue

        if "text" not in architecture.get("input_modalities", []):
            continue

        if "text" not in architecture.get("output_modalities", []):
            continue

        if model.get("context_length", 0) < min_context_length:
            continue

        if not required_parameters.issubset(supported_parameters):
            continue

        if reasoning.get("mandatory", False):
            continue

        candidates.append(
            {
                "id": model["id"],
                "name": model["name"],
                "context_length": model["context_length"],
                "max_completion_tokens": (
                    model.get("top_provider", {})
                    .get("max_completion_tokens")
                ),
                "supported_parameters": sorted(supported_parameters),
                "reasoning": reasoning,
            }
        )
        
    candidates.sort(
        key=lambda model: (
            model["reasoning"].get("default_enabled", False),
            -model["context_length"],
        )
    )

    return candidates[:k]

In [7]:
candidates = model_selection(
    required_parameters=REQUIRED_PARAMETERS,
    k=N_CANDIDATES,
    min_context_length=MIN_CONTEXT_LENGTH,
)

In [8]:
model = candidates[0]
model_name = model["id"]

model_name

'apodex/apodex-1.1-mini:free'

In [9]:
print(f"Модель: {model_name}")
print(f"context_length: {model['context_length']}")
print(f"max_completion_tokens: {model['max_completion_tokens']}")
print(f"reasoning: {model['reasoning']}")
print(
    "required_parameters:",
    REQUIRED_PARAMETERS.issubset(
        set(model['supported_parameters'])
    )
)

Модель: apodex/apodex-1.1-mini:free
context_length: 262144
max_completion_tokens: 235929
reasoning: {'mandatory': False}
required_parameters: True


# Тестовый запрос

In [10]:
api_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    timeout=120,
)

In [11]:
dev_data = load_jsonl("../data/dev.jsonl")

LOCAL_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)

In [12]:
test_product = prepare_product_input(
    product=dev_data[0],
    tokenizer=tokenizer,
    max_input_tokens=1500,
)

messages = build_messages(test_product)

response = await api_client.chat.completions.create(
    model=model_name,
    messages=messages,
    temperature=0.5,
    top_p=0.9,
    max_tokens=250,
    seed=42,
    response_format={"type": "json_object"},
    extra_body={
        "repetition_penalty": 1.1,
        "reasoning": {
            "enabled": False,
        },
    },
)

content = response.choices[0].message.content
finish_reason = response.choices[0].finish_reason

print("finish_reason:", finish_reason)

if response.usage is not None:
    print("prompt_tokens:", response.usage.prompt_tokens)
    print("completion_tokens:", response.usage.completion_tokens)
    print("total_tokens:", response.usage.total_tokens)

print("\nОтвет модели:")
print(content)

finish_reason: stop
prompt_tokens: 1048
completion_tokens: 196
total_tokens: 1244

Ответ модели:
{
  "description": "Пылесос Klin Eco с контейнером объёмом 2 л и мощностью всасывания 220 Вт. Весит всего 1.5 кг, что удобно для длительной уборки. Работает до 60 минут от сети.",
  "pros": [
    "Лёгкий",
    "Тихий",
    "Удобные насадки",
    "Быстрая очистка контейнера"
  ],
  "cons": [
    "Щётка забивается волосами"
  ],
  "tags": [
    "пылесос",
    "Klin",
    "с контейнером",
    "лёгкий пылесос",
    "тихий пылесос",
    "для шерсти животных",
    "мощность 220 Вт"
  ]
}


In [13]:
try:
    card = parse_and_validate(
        content=content,
        source_product=test_product,
    )

    print("Карточка валидна:")
    print(card.model_dump())

except Exception as error:
    print("Карточка не прошла валидацию:")
    print(error)

Карточка валидна:
{'product_id': 'P-0027', 'description': 'Пылесос Klin Eco с контейнером объёмом 2 л и мощностью всасывания 220 Вт. Весит всего 1.5 кг, что удобно для длительной уборки. Работает до 60 минут от сети.', 'pros': ['Лёгкий', 'Тихий', 'Удобные насадки', 'Быстрая очистка контейнера'], 'cons': ['Щётка забивается волосами'], 'tags': ['пылесос', 'Klin', 'с контейнером', 'лёгкий пылесос', 'тихий пылесос', 'для шерсти животных', 'мощность 220 Вт']}


In [14]:
message = response.choices[0].message

print(
    "reasoning:",
    getattr(message, "reasoning", None),
)

print(
    "reasoning_details:",
    getattr(message, "reasoning_details", None),
)

reasoning: None
reasoning_details: None


# Один запрос

In [15]:
def calculate_retry_delay(
        attempt: int,
        base_delay: float = 1.0,
        max_delay: float = 30.0
) -> float:
    """Рассчитывает задержку перед повторным API-запросом."""

    exponential_delay = base_delay * 2 ** (attempt - 1)
    jitter = random.uniform(0, base_delay)

    return min(exponential_delay + jitter, max_delay)

In [16]:
async def create_chat_completion_with_retry(
        client: AsyncOpenAI,
        model_name: str,
        messages: list[dict],
        generation_params: dict,
        max_api_attempts: int = 4,
        base_delay: float = 1.0,
        max_delay: float = 30.0
):
    """Отправляет API-запрос с повторами при временных ошибках."""

    retryable_errors = (
        RateLimitError,
        APITimeoutError,
        APIConnectionError,
        InternalServerError,
    )

    api_params = generation_params.copy()

    seed = api_params.pop("seed", 42)
    repetition_penalty = api_params.pop(
        "repetition_penalty",
        1.1,
    )

    for attempt in range(1, max_api_attempts + 1):
        try:
            return await client.chat.completions.create(
                model=model_name,
                messages=messages,
                response_format={"type": "json_object"},
                seed=seed,
                extra_body={
                    "repetition_penalty": repetition_penalty,
                    "reasoning": {
                        "enabled": False,
                    },
                },
                **api_params,
            )

        except retryable_errors:
            if attempt == max_api_attempts:
                raise

            delay = calculate_retry_delay(
                attempt=attempt,
                base_delay=base_delay,
                max_delay=max_delay,
            )

            print(
                f"Временная ошибка API. "
                f"Повтор {attempt + 1}/{max_api_attempts} "
                f"через {delay:.2f} с"
            )

            await asyncio.sleep(delay)

        except BadRequestError:
            raise

# Генерация одной карточки

In [17]:
async def generate_one_api(
        client: AsyncOpenAI,
        model_name: str,
        source_product: dict,
        generation_params: dict,
        max_attempts: int = 2
) -> GenerationResult:
    """Генерирует и валидирует одну карточку через OpenRouter."""

    messages = build_messages(source_product)

    all_errors = []
    input_tokens = 0
    output_tokens = 0

    base_seed = generation_params.get("seed", 42)

    for attempt in range(1, max_attempts + 1):
        current_params = generation_params.copy()
        current_params["seed"] = base_seed + attempt - 1

        try:
            response = await create_chat_completion_with_retry(
                client=client,
                model_name=model_name,
                messages=messages,
                generation_params=current_params,
                max_api_attempts=MAX_API_ATTEMPTS,
            )

        except BadRequestError as error:
            print(
                f"{source_product['product_id']}: "
                f"ошибка запроса 400: {error}"
            )

            return GenerationResult(
                product_id=source_product["product_id"],
                card=None,
                valid=False,
                attempts=attempt,
                errors=all_errors + [ValidationReason.OTHER],
                input_tokens=input_tokens,
                output_tokens=output_tokens,
            )

        except (
            RateLimitError,
            APITimeoutError,
            APIConnectionError,
            InternalServerError,
        ) as error:
            print(
                f"{source_product['product_id']}: "
                f"API недоступен после повторов: "
                f"{type(error).__name__}"
            )

            return GenerationResult(
                product_id=source_product["product_id"],
                card=None,
                valid=False,
                attempts=attempt,
                errors=all_errors + [ValidationReason.OTHER],
                input_tokens=input_tokens,
                output_tokens=output_tokens,
            )

        content = response.choices[0].message.content
        finish_reason = response.choices[0].finish_reason

        if response.usage is not None:
            input_tokens += response.usage.prompt_tokens or 0
            output_tokens += response.usage.completion_tokens or 0

        try:
            if finish_reason == "length":
                raise ResponseValidationError(
                    reasons=[
                        ValidationReason.OUTPUT_TRUNCATED
                    ],
                    message=(
                        "Ответ модели был обрезан "
                        "по max_tokens."
                    ),
                )

            card = parse_and_validate(
                content=content,
                source_product=source_product,
            )

            return GenerationResult(
                product_id=source_product["product_id"],
                card=card,
                valid=True,
                attempts=attempt,
                errors=all_errors,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
            )

        except ResponseValidationError as error:
            all_errors.extend(error.reasons)

            if attempt == max_attempts:
                break

            if ValidationReason.EMPTY_RESPONSE in error.reasons:
                messages = build_messages(source_product)
                continue

            messages = build_messages(source_product)

            if ValidationReason.OUTPUT_TRUNCATED in error.reasons:
                messages.append(
                    {
                        "role": "user",
                        "content": build_retry_message(error),
                    }
                )

            else:
                messages.extend(
                    [
                        {
                            "role": "assistant",
                            "content": content,
                        },
                        {
                            "role": "user",
                            "content": build_retry_message(error),
                        },
                    ]
                )

    return GenerationResult(
        product_id=source_product["product_id"],
        card=None,
        valid=False,
        attempts=max_attempts,
        errors=all_errors,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
    )

# Последовательная обработка

In [18]:
async def run_api_comparison(
        items: list[dict],
        client: AsyncOpenAI,
        model_name: str,
        generation_params: dict,
) -> tuple[list[tuple[GenerationResult, float]], float]:
    """Последовательно обрабатывает набор товаров через OpenRouter."""

    results = []

    started = time.perf_counter()

    for index, product in enumerate(items, start=1):
        item_started = time.perf_counter()

        result = await generate_one_api(
            client=client,
            model_name=model_name,
            source_product=product,
            generation_params=generation_params,
            max_attempts=MAX_ATTEMPTS,
        )

        latency = time.perf_counter() - item_started

        results.append(
            (result, latency)
        )

        print(
            f"{index}/{len(items)} "
            f"{result.product_id}: "
            f"valid={result.valid}, "
            f"attempts={result.attempts}, "
            f"latency={latency:.2f} с"
        )

    elapsed = time.perf_counter() - started

    return results, elapsed

In [19]:
api_products = [
    prepare_product_input(
        product=product,
        tokenizer=tokenizer,
        max_input_tokens=1500,
    )
    for product in dev_data[:API_SAMPLE_SIZE]
]

In [20]:
api_results, api_elapsed = await run_api_comparison(
    items=api_products,
    client=api_client,
    model_name=model_name,
    generation_params=GENERATION_PARAMS,
)

1/20 P-0027: valid=True, attempts=1, latency=2.17 с
2/20 P-0032: valid=True, attempts=1, latency=1.89 с
3/20 P-0017: valid=True, attempts=1, latency=1.53 с
4/20 P-0021: valid=True, attempts=2, latency=3.15 с
5/20 P-0036: valid=True, attempts=1, latency=1.76 с
6/20 P-0039: valid=True, attempts=2, latency=3.84 с
7/20 P-0024: valid=True, attempts=1, latency=1.87 с
8/20 P-0009: valid=True, attempts=1, latency=2.16 с
9/20 P-0037: valid=True, attempts=1, latency=1.67 с
10/20 P-0002: valid=True, attempts=2, latency=4.24 с
11/20 P-0016: valid=True, attempts=1, latency=1.89 с
12/20 P-0028: valid=True, attempts=1, latency=2.03 с
13/20 P-0012: valid=True, attempts=1, latency=1.97 с
14/20 P-0020: valid=True, attempts=1, latency=1.88 с
15/20 P-0026: valid=True, attempts=1, latency=2.05 с
16/20 P-0003: valid=True, attempts=1, latency=2.01 с
17/20 P-0035: valid=True, attempts=1, latency=2.33 с
18/20 P-0007: valid=True, attempts=1, latency=1.94 с
19/20 P-0022: valid=True, attempts=1, latency=1.81 с
20

# Метрики для API

In [21]:
from collections import Counter


API_INPUT_PRICE = 20
API_OUTPUT_PRICE = 60


def calculate_api_metrics(
        api_results,
        elapsed_sec: float
) -> dict:
    """Вычисляет метрики прогона через внешний API."""

    results = [
        result
        for result, _ in api_results
    ]

    latencies = [
        latency
        for _, latency in api_results
    ]

    n = len(results)

    valid = sum(
        result.valid
        for result in results
    )

    valid_rate = (
        valid / n
        if n
        else 0.0
    )

    input_tokens = sum(
        result.input_tokens
        for result in results
    )

    output_tokens = sum(
        result.output_tokens
        for result in results
    )

    avg_input_tokens = (
        input_tokens / n
        if n
        else 0.0
    )

    avg_output_tokens = (
        output_tokens / n
        if n
        else 0.0
    )

    avg_latency = (
        sum(latencies) / len(latencies)
        if latencies
        else 0.0
    )

    avg_attempts = (
        sum(
            result.attempts
            for result in results
        ) / n
        if n
        else 0.0
    )

    errors_by_reason = Counter(
        error.value
        for result in results
        if not result.valid
        for error in set(result.errors)
    )

    cost_per_1000 = (
        (
            input_tokens * API_INPUT_PRICE
            + output_tokens * API_OUTPUT_PRICE
        )
        / 1_000_000
        / n
        * 1000
        if n
        else 0.0
    )

    return {
        "model": model_name,
        "n": n,
        "valid": valid,
        "valid_rate": valid_rate,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "elapsed_sec": elapsed_sec,
        "avg_latency": avg_latency,
        "avg_attempts": avg_attempts,
        "cost_per_1000": cost_per_1000,
        "errors_by_reason": dict(errors_by_reason),
    }

In [22]:
api_metrics = calculate_api_metrics(
    api_results=api_results,
    elapsed_sec=api_elapsed,
)

api_metrics

{'model': 'apodex/apodex-1.1-mini:free',
 'n': 20,
 'valid': 20,
 'valid_rate': 1.0,
 'input_tokens': 25528,
 'output_tokens': 4396,
 'avg_input_tokens': 1276.4,
 'avg_output_tokens': 219.8,
 'elapsed_sec': 43.94818579899993,
 'avg_latency': 2.1973266291500066,
 'avg_attempts': 1.15,
 'cost_per_1000': 38.716,
 'errors_by_reason': {}}

# Локальный прогон (последовательная обработка)

In [23]:
from generation import generate_one
from io_utils import save_json, save_jsonl
from check_metrics import (
    calculate_run_stats,
    calculate_metrics,
)

In [24]:
local_client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=120
)

In [25]:
async def run_local(
        items: list[dict],
        client: AsyncOpenAI,
        model_name: str,
        generation_params: dict,
        concurrency: int
) -> tuple[list[tuple[GenerationResult, float]], float]:
    """Обрабатывает набор товаров через локальный vLLM."""

    semaphore = asyncio.Semaphore(concurrency)

    async def generate(product):
        async with semaphore:
            started = time.perf_counter()

            result = await generate_one(
                client=client,
                model_name=model_name,
                source_product=product,
                generation_params=generation_params,
                max_attempts=MAX_ATTEMPTS,
            )

            latency = time.perf_counter() - started

            return result, latency

    started = time.perf_counter()

    results = await asyncio.gather(
        *[
            generate(product)
            for product in items
        ]
    )

    elapsed = time.perf_counter() - started

    return results, elapsed

In [26]:
warmup_product = api_products[0]

warmup_result, warmup_latency = await run_local(
    items=[warmup_product],
    client=local_client,
    model_name=LOCAL_MODEL_NAME,
    generation_params=GENERATION_PARAMS,
    concurrency=1,
)

print(f"Прогрев завершён, latency: {warmup_latency:.2f} с")

Прогрев завершён, latency: 18.85 с


In [27]:
local_results, local_elapsed = await run_local(
        items=api_products,
        client=local_client,
        model_name=LOCAL_MODEL_NAME,
        generation_params=GENERATION_PARAMS,
        concurrency=1
)

In [28]:
local_results_only = [
    result
    for result, _ in local_results
]

local_latencies = [

    
    latency
    for _, latency in local_results
]

local_n = len(local_results_only)

local_valid = sum(
    result.valid
    for result in local_results_only
)

local_input_tokens = sum(
    result.input_tokens
    for result in local_results_only
)

local_output_tokens = sum(
    result.output_tokens
    for result in local_results_only
)

local_comparison_metrics = {
    "model": LOCAL_MODEL_NAME,
    "n": local_n,
    "valid": local_valid,
    "valid_rate": (
        local_valid / local_n
        if local_n
        else 0.0
    ),
    "input_tokens": local_input_tokens,
    "output_tokens": local_output_tokens,
    "avg_input_tokens": (
        local_input_tokens / local_n
        if local_n
        else 0.0
    ),
    "avg_output_tokens": (
        local_output_tokens / local_n
        if local_n
        else 0.0
    ),
    "elapsed_sec": local_elapsed,
    "avg_latency": (
        sum(local_latencies) / len(local_latencies)
        if local_latencies
        else 0.0
    ),
    "avg_attempts": (
        sum(
            result.attempts
            for result in local_results_only
        ) / local_n
        if local_n
        else 0.0
    ),
}

local_comparison_metrics

{'model': 'Qwen/Qwen2.5-1.5B-Instruct',
 'n': 20,
 'valid': 17,
 'valid_rate': 0.85,
 'input_tokens': 36897,
 'output_tokens': 5557,
 'avg_input_tokens': 1844.85,
 'avg_output_tokens': 277.85,
 'elapsed_sec': 110.2642608030003,
 'avg_latency': 5.513167085949954,
 'avg_attempts': 1.45}

# Сравнение

In [29]:
comparison_df = pd.DataFrame(
    {
        "Локальная модель": {
            "Модель": local_comparison_metrics["model"],
            "Количество карточек": local_comparison_metrics["n"],
            "Доля валидных": local_comparison_metrics["valid_rate"],
            "Входных токенов / карточку": local_comparison_metrics["avg_input_tokens"],
            "Выходных токенов / карточку": local_comparison_metrics["avg_output_tokens"],
            "Средняя latency, с": local_comparison_metrics["avg_latency"],
            "Среднее число попыток": local_comparison_metrics["avg_attempts"],
        },
        "OpenRouter": {
            "Модель": api_metrics["model"],
            "Количество карточек": api_metrics["n"],
            "Доля валидных": api_metrics["valid_rate"],
            "Входных токенов / карточку": api_metrics["avg_input_tokens"],
            "Выходных токенов / карточку": api_metrics["avg_output_tokens"],
            "Средняя latency, с": api_metrics["avg_latency"],
            "Среднее число попыток": api_metrics["avg_attempts"],
        },
    }
)

comparison_df

,Локальная модель,OpenRouter
Модель,Qwen/Qwen2.5-1.5B-Instruct,apodex/apodex-1.1-mini:free
Количество карточек,20,20
Доля валидных,0.85,1.0
Входных токенов / карточку,1844.85,1276.4
Выходных токенов / карточку,277.85,219.8
"Средняя latency, с",5.513167,2.197327
Среднее число попыток,1.45,1.15


# Финальный прогон

In [30]:
benchmark = load_jsonl("../data/benchmark.jsonl")

prepared_benchmark = [
    prepare_product_input(
        product=product,
        tokenizer=tokenizer,
        max_input_tokens=1500,
    )
    for product in benchmark
]

len(prepared_benchmark)

300

In [31]:
final_results, final_elapsed = await run_local(
    items=prepared_benchmark,
    client=local_client,
    model_name=LOCAL_MODEL_NAME,
    generation_params=GENERATION_PARAMS,
    concurrency=48
)

In [32]:
final_run_stats = calculate_run_stats(
    run_results=final_results,
    elapsed_sec=final_elapsed,
    model_name=LOCAL_MODEL_NAME,
    concurrency=48,
    generation_params=GENERATION_PARAMS,
)

final_metrics = calculate_metrics(
    run_results=final_results,
    elapsed_sec=final_elapsed,
)

final_run_stats

{'model': 'Qwen/Qwen2.5-1.5B-Instruct',
 'concurrency': 48,
 'params': {'temperature': 0.5,
  'top_p': 0.9,
  'repetition_penalty': 1.1,
  'max_tokens': 250,
  'seed': 42},
 'elapsed_sec': 190.9048873920001,
 'input_tokens': 466128,
 'output_tokens': 74463,
 'n': 300,
 'avg_latency': 29.11005132916333,
 'avg_attempts': 1.2533333333333334}

In [33]:
final_metrics

{'valid': 271,
 'valid_rate': 0.9033333333333333,
 'throughput': 1.5714631725692088,
 'time_for_300_sec': 190.9048873920001,
 'cost_vllm_per_1000': 11.489646000444449,
 'errors_by_reason': {'description_sentence_count': 5,
  'description_length': 9,
  'wrong_list_length': 16,
  'output_truncated': 14}}

In [34]:
predictions = []

for result, _ in final_results:
    if result.valid:
        predictions.append(
            result.card.model_dump()
        )

    else:
        predictions.append(
            {
                "product_id": result.product_id,
                "valid": False,
                "attempts": result.attempts,
                "errors": [
                    error.value
                    for error in result.errors
                ],
            }
        )

len(predictions)

300

In [35]:
save_jsonl(
    "../outputs/predictions.jsonl",
    predictions,
)